# Getting Started: Geometry Override for H2

This notebook shows how to run a calculation with an explicit geometry instead
of relying only on the built-in molecule registry.

Goals:

- inspect the default `H2` geometry from the registry
- define a custom `H2` geometry explicitly
- run VQE on both setups
- compare the returned energies

This is the basic workflow for moving from registry-based molecules to custom
user-specified geometries.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from common.hamiltonian import build_hamiltonian, get_exact_spectrum
from common.molecules import MOLECULES
from vqe.core import run_vqe

## Registry geometry

The package includes a built-in molecule registry. We start by inspecting the
stored entry for `H2`.

In [ ]:
MOLECULES["H2"]

The registry entry gives the default symbols and coordinates used when we call
high-level workflows like `run_vqe(molecule="H2", ...)` without overriding the
geometry.

In [ ]:
registry_symbols = MOLECULES["H2"]["symbols"]
registry_coordinates = np.asarray(MOLECULES["H2"]["coordinates"], dtype=float)
registry_basis = MOLECULES["H2"].get("basis", "sto-3g")
registry_charge = MOLECULES["H2"].get("charge", 0)
registry_unit = MOLECULES["H2"].get("unit", "angstrom")

print("Registry symbols     :", registry_symbols)
print("Registry coordinates :")
print(registry_coordinates)
print("Registry basis       :", registry_basis)
print("Registry charge      :", registry_charge)
print("Registry unit        :", registry_unit)

## Define an explicit geometry

Now we define the same molecule directly. This is the pattern to use when:

- you want a different bond length
- you want to sweep geometries
- your system is not one of the pre-registered molecules

Here we choose a simple custom `H2` geometry.

In [ ]:
custom_symbols = ["H", "H"]
custom_coordinates = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.80, 0.0, 0.0],
    ],
    dtype=float,
)

custom_basis = "sto-3g"
custom_charge = 0
custom_unit = "angstrom"

print("Custom symbols     :", custom_symbols)
print("Custom coordinates :")
print(custom_coordinates)

## Visual comparison of bond lengths

In [ ]:
registry_distance = float(np.linalg.norm(registry_coordinates[1] - registry_coordinates[0]))
custom_distance = float(np.linalg.norm(custom_coordinates[1] - custom_coordinates[0]))

print(f"Registry H-H distance: {registry_distance:.6f} {registry_unit}")
print(f"Custom   H-H distance: {custom_distance:.6f} {custom_unit}")

## Build Hamiltonians directly

Before running VQE, it is useful to confirm that both setups can be passed
through the Hamiltonian builder.

In [ ]:
H_reg, qubits_reg, hf_reg, symbols_reg, coords_reg, basis_reg, charge_reg, unit_reg = build_hamiltonian(
    molecule="H2",
    mapping="jordan_wigner",
)

H_custom, qubits_custom, hf_custom, symbols_custom_out, coords_custom_out, basis_custom_out, charge_custom_out, unit_custom_out = build_hamiltonian(
    molecule=None,
    symbols=custom_symbols,
    coordinates=custom_coordinates,
    basis=custom_basis,
    charge=custom_charge,
    unit=custom_unit,
    mapping="jordan_wigner",
)

print("Registry qubits:", qubits_reg)
print("Custom   qubits:", qubits_custom)

The physical geometry changed, but the workflow boundary stayed the same:

- build a Hamiltonian
- prepare the Hartree-Fock reference
- run the chosen algorithm

## Exact reference energies

For the registry molecule, we can compare against the exact spectrum using the
built-in helper.

In [ ]:
exact_registry_spectrum = np.asarray(get_exact_spectrum("H2"), dtype=float)
exact_registry_spectrum = np.sort(exact_registry_spectrum)
exact_registry_ground_energy = float(exact_registry_spectrum[0])

print("Registry exact spectrum:")
print(exact_registry_spectrum)
print()
print(f"Registry exact ground-state energy: {exact_registry_ground_energy:.10f}")

For the custom geometry, we can still build the Hamiltonian directly, but here
we focus on comparing the returned VQE energies rather than constructing a full
exact-spectrum helper workflow around the custom geometry.

## Run VQE with the registry geometry

In [ ]:
res_registry = run_vqe(
    molecule="H2",
    ansatz_name="UCCSD",
    optimizer_name="Adam",
    steps=50,
    stepsize=0.2,
    seed=0,
    noisy=False,
    force=True,
    plot=False,
)

In [ ]:
energy_registry = float(res_registry["energy"])
energies_registry = np.asarray(res_registry["energies"], dtype=float)

print(f"Registry VQE final energy: {energy_registry:.10f}")

## Run VQE with the explicit geometry

We now pass the geometry directly into `run_vqe(...)`.

In [ ]:
res_custom = run_vqe(
    molecule=None,
    symbols=custom_symbols,
    coordinates=custom_coordinates,
    basis=custom_basis,
    charge=custom_charge,
    unit=custom_unit,
    ansatz_name="UCCSD",
    optimizer_name="Adam",
    steps=50,
    stepsize=0.2,
    seed=0,
    noisy=False,
    force=True,
    plot=False,
)

In [ ]:
energy_custom = float(res_custom["energy"])
energies_custom = np.asarray(res_custom["energies"], dtype=float)

print(f"Custom geometry VQE final energy: {energy_custom:.10f}")

## Compare the two runs

Changing the geometry changes the Hamiltonian, so the final energy changes too.

In [ ]:
print(f"{'Case':<22} {'Bond length':>14} {'Final energy':>18}")
print("-" * 58)
print(f"{'Registry H2':<22} {registry_distance:>14.6f} {energy_registry:>18.10f}")
print(f"{'Custom geometry H2':<22} {custom_distance:>14.6f} {energy_custom:>18.10f}")

## Energy trajectories

We can also compare the optimization traces directly.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(energies_registry)), energies_registry, marker="o", label="Registry H2")
plt.plot(np.arange(len(energies_custom)), energies_custom, marker="o", label="Custom H2 geometry")
plt.axhline(exact_registry_ground_energy, linestyle="--", label="Registry exact ground energy")
plt.xlabel("Iteration")
plt.ylabel("Energy [Ha]")
plt.title("VQE with registry vs explicit H2 geometry")
plt.grid(True)
plt.legend()
plt.show()

## What changed and what did not

What changed:

- the nuclear coordinates
- the resulting electronic Hamiltonian
- the final variational energy

What did not change:

- the high-level `run_vqe(...)` workflow
- the ansatz / optimizer interface
- the general plotting and result structure

## Interpretation

The built-in molecule registry is convenient for quick examples, but explicit
geometry overrides are what make the package flexible for real studies.

This same pattern extends naturally to:

- bond scans
- angle scans
- custom molecular systems
- non-registry geometries used in method comparisons

## What this notebook showed

We:

- inspected the default `H2` registry entry
- defined a custom `H2` geometry explicitly
- built both Hamiltonians
- ran VQE on both setups
- compared the returned energies and trajectories

This is the basic geometry-override workflow in the repository.

## Next steps

Good follow-ups are:

- scan several H-H distances and plot an energy curve
- compare different ansätze on the same custom geometry
- use the same pattern for `LiH`, `H2O`, or another system